# Linux & Shell Scripting for DevOps

## Linux Fundamentals for DevOps

DevOps engineers need to be comfortable with Linux command-line tools and system administration. Key areas include:

- **File system navigation**: `cd`, `ls`, `find`, `locate`
- **File operations**: `cp`, `mv`, `rm`, `chmod`, `chown`
- **Text processing**: `grep`, `sed`, `awk`, `cut`, `sort`
- **Process management**: `ps`, `top`, `kill`, `systemctl`
- **Networking**: `ifconfig`, `netstat`, `ss`, `curl`, `wget`
- **Package management**: `apt`, `yum`, `dnf` depending on distribution

## Bash Scripting Essentials

Bash scripts automate infrastructure tasks. A well-written script is idempotent, handles errors gracefully, and logs its actions.

```bash
#!/bin/bash
set -euo pipefail  # Exit on error, undefined vars, pipe failures

# Logging function
log() {
  echo "[$(date +'%Y-%m-%d %H:%M:%S')] $*" | tee -a /var/log/script.log
}

# Error handling
trap 'log "Error on line $LINENO"' ERR

log "Starting deployment script"

# Check prerequisites
if ! command -v docker &> /dev/null; then
  log "ERROR: Docker not installed"
  exit 1
fi

# Main logic
docker pull myapp:latest
docker run -d --name myapp myapp:latest

log "Deployment completed successfully"
```

## Cron Jobs for Scheduled Tasks

Cron allows scheduling scripts to run at specific times. The crontab format is: `minute hour day month weekday command`

```bash
# View current crontab
crontab -l

# Edit crontab
crontab -e

# Example crontab entries
0 2 * * * /usr/local/bin/backup.sh              # Daily at 2 AM
*/15 * * * * /usr/local/bin/health-check.sh     # Every 15 minutes
0 0 * * 0 /usr/local/bin/weekly-report.sh       # Weekly on Sunday at midnight
```

## Systemd Services

Systemd manages services on modern Linux systems. DevOps engineers often create custom systemd units for applications.

```bash
# Create a systemd service file
cat > /etc/systemd/system/myapp.service << 'EOF'
[Unit]
Description=My Application
After=network.target

[Service]
Type=simple
User=appuser
WorkingDirectory=/opt/myapp
ExecStart=/opt/myapp/bin/start.sh
Restart=on-failure
RestartSec=10

[Install]
WantedBy=multi-user.target
EOF

# Enable and start the service
systemctl daemon-reload
systemctl enable myapp
systemctl start myapp
systemctl status myapp
```

## Advanced Bash Patterns

### Error Handling and Validation

```bash
#!/bin/bash
set -euo pipefail

# Function with error handling
deploy_app() {
  local app_name=$1
  local version=$2
  
  if [[ -z "$app_name" || -z "$version" ]]; then
    echo "Usage: deploy_app <app_name> <version>"
    return 1
  fi
  
  echo "Deploying $app_name:$version"
  # Deployment logic here
}

# Validate input
if [[ $# -lt 2 ]]; then
  echo "Error: Missing arguments"
  exit 1
fi

deploy_app "$@"
```

### Parallel Execution

```bash
#!/bin/bash

# Run multiple tasks in parallel
run_tests() {
  local test_dirs=("unit" "integration" "e2e")
  
  for dir in "${test_dirs[@]}"; do
    (
      cd "tests/$dir"
      ./run.sh
    ) &
  done
  
  # Wait for all background jobs
  wait
  echo "All tests completed"
}

run_tests
```

## AWS CLI Scripting

The AWS CLI is essential for DevOps automation on AWS.

```bash
#!/bin/bash

# List all EC2 instances
aws ec2 describe-instances \
  --query 'Reservations[].Instances[].{ID:InstanceId,Type:InstanceType,State:State.Name}' \
  --output table

# Create an S3 bucket with versioning
aws s3api create-bucket \
  --bucket my-backup-bucket \
  --region us-east-1

aws s3api put-bucket-versioning \
  --bucket my-backup-bucket \
  --versioning-configuration Status=Enabled

# Get CloudWatch metrics
aws cloudwatch get-metric-statistics \
  --namespace AWS/EC2 \
  --metric-name CPUUtilization \
  --dimensions Name=InstanceId,Value=i-1234567890abcdef0 \
  --start-time 2024-01-01T00:00:00Z \
  --end-time 2024-01-02T00:00:00Z \
  --period 3600 \
  --statistics Average
```

## Text Processing with grep, sed, awk

These tools are powerful for parsing logs and configuration files.

```bash
# grep: Find lines matching a pattern
grep "ERROR" /var/log/app.log
grep -i "warning" /var/log/app.log  # Case-insensitive
grep -c "ERROR" /var/log/app.log    # Count matches

# sed: Stream editor for text transformation
sed 's/old/new/g' config.txt        # Replace all occurrences
sed -i.bak 's/old/new/g' config.txt # In-place edit with backup

# awk: Text processing and data extraction
awk '{print $1, $3}' data.txt       # Print columns 1 and 3
awk -F: '{print $1}' /etc/passwd    # Use colon as delimiter
awk '$2 > 100 {print $1}' data.txt  # Conditional printing
```

## Monitoring and Logging

DevOps scripts should produce clear logs for troubleshooting.

```bash
#!/bin/bash

LOG_FILE="/var/log/deployment.log"
ERROR_LOG="/var/log/deployment-errors.log"

log_info() {
  echo "[INFO] $(date '+%Y-%m-%d %H:%M:%S') - $*" | tee -a "$LOG_FILE"
}

log_error() {
  echo "[ERROR] $(date '+%Y-%m-%d %H:%M:%S') - $*" | tee -a "$ERROR_LOG" >&2
}

# Usage
log_info "Starting deployment"
if ! docker pull myapp:latest; then
  log_error "Failed to pull Docker image"
  exit 1
fi
log_info "Deployment completed"
```

---

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What does `set -euo pipefail` do in a bash script?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q2847591" value="0">
      <span>Exit on error, undefined variables, and pipe failures</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q2847591" value="1">
      <span>Enable debugging mode</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q2847591" value="2">
      <span>Set environment variables</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q2847591" value="3">
      <span>Enable parallel execution</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What cron expression runs a job every 30 minutes?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5739284" value="0">
      <span>30 * * * *</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5739284" value="1">
      <span>* 30 * * *</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5739284" value="2">
      <span>*/30 * * * *</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5739284" value="3">
      <span>0 */30 * * *</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ How do you enable a systemd service to start on boot?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8361947" value="0">
      <span>systemctl start myapp</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8361947" value="1">
      <span>systemctl enable myapp</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8361947" value="2">
      <span>systemctl activate myapp</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8361947" value="3">
      <span>systemctl daemon-reload</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What does the `grep -c` command do?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7194528" value="0">
      <span>Count the number of matching lines</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7194528" value="1">
      <span>Display matching lines with context</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7194528" value="2">
      <span>Case-insensitive search</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7194528" value="3">
      <span>Compile the pattern</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ Which AWS CLI command lists EC2 instances?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9482736" value="0">
      <span>aws ec2 list-instances</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9482736" value="1">
      <span>aws ec2 get-instances</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9482736" value="2">
      <span>aws ec2 describe-instances</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9482736" value="3">
      <span>aws ec2 show-instances</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>